#Import Required Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

#Load and Prepare Dataset

In [ ]:
# Load the dataset
data = fetch_california_housing(as_frame=True)
df = pd.concat([data.data, data.target.rename("HousePrice")], axis=1)

# Separate features and target
X = df.drop("HousePrice", axis=1)
y = df["HousePrice"]

df.head()

#Feature Scaling

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

#Detect Overfitting (Train vs Test Performance)

In [ ]:
# Train an unconstrained Decision Tree
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)

# Make predictions on both sets
train_pred = tree.predict(X_train)
test_pred = tree.predict(X_test)

# Calculate RMSE
train_rmse = mean_squared_error(y_train, train_pred, squared=False)
test_rmse = mean_squared_error(y_test, test_pred, squared=False)

print(f"Training RMSE: {train_rmse:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")

#Cross-Validation (Reliable Evaluation)

In [ ]:
# Evaluate the model using 5-Fold Cross-Validation
cv_scores = cross_val_score(
    tree, X_scaled, y,
    scoring="neg_root_mean_squared_error",
    cv=5
)

# Convert negative RMSE scores to positive and find the mean
cv_rmse = -cv_scores.mean()

print(f"Cross-Validation RMSE: {cv_rmse:.4f}")

#Hyperparameter Tuning Using GridSearchCV

In [ ]:
# Define the hyperparameters grid to search
param_grid = {
    "max_depth": [3, 5, 7, 10],
    "min_samples_split": [2, 5, 10]
}

# Set up GridSearchCV
grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5
)

# Fit the grid search to the training data
grid.fit(X_train, y_train)

print(f"Best parameters found: {grid.best_params_}")

#Evaluate Optimized Model

In [ ]:
# Extract the best model from the grid search
best_tree = grid.best_estimator_

# Make predictions on the test set
y_pred = best_tree.predict(X_test)

# Evaluate metrics
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

print(f"Tuned Decision Tree RMSE: {rmse:.4f}")
print(f"Tuned Decision Tree R2 Score: {r2:.4f}")

#Model Comparison Summary Table

In [ ]:
# Train Baseline Models for Comparison
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_rmse = mean_squared_error(y_test, lr_pred, squared=False)
lr_r2 = r2_score(y_test, lr_pred)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)
ridge_rmse = mean_squared_error(y_test, ridge_pred, squared=False)
ridge_r2 = r2_score(y_test, ridge_pred)

# Create comparison table
results = {
    "Model": ["Linear Regression", "Ridge Regression", "Tuned Decision Tree"],
    "RMSE": [lr_rmse, ridge_rmse, rmse],
    "R2 Score": [lr_r2, ridge_r2, r2]
}

results_df = pd.DataFrame(results)
results_df

### Final Model Selection Justification

* **Why this model was selected:** The Tuned Decision Tree was chosen because it achieved a better balance of predictive accuracy (lower RMSE) and explanatory power (higher R²) compared to the baseline Linear and Ridge Regression models.
* **How overfitting was reduced:** By restricting the hyperparameters (`max_depth` and `min_samples_split`) via GridSearchCV, the Decision Tree was prevented from memorizing the training data, thereby controlling overfitting.
* **Why cross-validation results are trusted:** 5-fold cross-validation evaluates the model across multiple different subsets of the data, providing a much more stable and realistic estimate of performance than a single train-test split.
* **Trade-offs between simplicity and performance:** While a simple Linear Regression is easier to interpret, trading that simplicity for a tuned Decision Tree allows the model to capture non-linear relationships, resulting in stronger generalization and predictive performance on unseen data.